1. GENERACIÓN DE DATA SIMULADA (FEATURES)

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler

np.random.seed(42)

# Features Clientes: [Gusto Acción, Gusto Drama, Gusto Sci-Fi, Es Premium]
clientes_features = pd.DataFrame({
    'user_id': [1, 2, 3, 4, 5],
    'f_accion': [0.9, 0.1, 0.2, 0.8, 0.5],
    'f_drama':  [0.1, 0.8, 0.9, 0.2, 0.5],
    'f_scifi':  [0.8, 0.2, 0.1, 0.9, 0.5],
    'es_premium': [1, 0, 1, 1, 0]
})

# Features Películas: [Es Acción, Es Drama, Es Sci-Fi, Costo Licencia]
peliculas_features = pd.DataFrame({
    'movie_id': [101, 102, 103, 104, 105, 106],
    'f_accion': [1, 0, 0, 1, 0, 1],
    'f_drama':  [0, 1, 1, 0, 1, 0],
    'f_scifi':  [1, 0, 0, 1, 0, 0],
    'precio_alquiler': [5.99, 3.99, 2.99, 4.99, 3.50, 1.99] # Valor monetario
})



2. SIMILARIDAD Y PREDICCIÓN (RANKING)


In [3]:

# Calculamos similitud coseno entre gustos de usuarios y géneros de películas
user_v = clientes_features[['f_accion', 'f_drama', 'f_scifi']].values
item_v = peliculas_features[['f_accion', 'f_drama', 'f_scifi']].values
sim_matrix = cosine_similarity(user_v, item_v)

# Convertimos a un DataFrame de "Probabilidades"
preds = []
for i, user_id in enumerate(clientes_features['user_id']):
    for j, movie_id in enumerate(peliculas_features['movie_id']):
        preds.append({
            'user_id': user_id,
            'movie_id': movie_id,
            'prob_ia': sim_matrix[i][j],
            'precio': peliculas_features.iloc[j]['precio_alquiler']
        })

df_rank = pd.DataFrame(preds)

3. RE-RANKING (REGLAS DE NEGOCIO)

In [4]:

# Regla 1: Si el usuario es Premium, boost a películas caras (queremos que use su suscripción)
df_rank = df_rank.merge(clientes_features[['user_id', 'es_premium']], on='user_id')

def apply_business_rules(row):
    score = row['prob_ia']
    # Boost por usuario premium en películas de estreno (precio > 4)
    if row['es_premium'] == 1 and row['precio'] > 4:
        score += 0.1
    # Penalización si la probabilidad es muy baja (no mostrar basura)
    if score < 0.3:
        score -= 0.5
    return score

df_rank['final_score'] = df_rank.apply(apply_business_rules, axis=1)



4. MÉTRICAS @K (nDCG, RECALL, PRECISION)

In [5]:

# Simulamos "Gusto Real" (Ground Truth) para evaluar
df_rank['real_interest'] = np.where(df_rank['prob_ia'] > 0.85, 1, 0)

def calculate_metrics(user_id):
    user_data = df_rank[df_rank['user_id'] == user_id].sort_values('final_score', ascending=False).head(5)
    hits = user_data['real_interest'].sum()

    precision_5 = hits / 5
    # Supongamos que hay 3 películas que realmente le gustan al usuario en total
    recall_5 = hits / 3

    # nDCG simplificado
    dcg = sum([rel / np.log2(idx + 2) for idx, rel in enumerate(user_data['real_interest'])])
    idcg = sum([1 / np.log2(idx + 2) for idx in range(min(hits, 5))])
    ndcg = dcg / idcg if idcg > 0 else 0

    return precision_5, recall_5, ndcg

# Evaluar para el Usuario 1
p5, r5, n = calculate_metrics(1)



5. VALOR MONETARIO (ROI)

In [6]:

# Calculamos el valor esperado: Probabilidad x Precio
df_rank['valor_esperado'] = df_rank['final_score'] * df_rank['precio']
total_revenue_est = df_rank.groupby('user_id')['valor_esperado'].sum().sum()



6. Resultado final

In [7]:
print(f"--- RECOMENDACIONES TOP 5 (USUARIO 1) ---")
print(df_rank[df_rank['user_id'] == 1].sort_values('final_score', ascending=False).head(5))
print(f"\n--- MÉTRICAS DE CALIDAD ---")
print(f"Precision@5: {p5:.2f}")
print(f"Recall@5:    {r5:.2f}")
print(f"nDCG@5:      {n:.2f}")
print(f"\n--- IMPACTO DE NEGOCIO ---")
print(f"Ingreso Estimado Total (Simulado): ${total_revenue_est:.2f}")

--- RECOMENDACIONES TOP 5 (USUARIO 1) ---
   user_id  movie_id   prob_ia  precio  es_premium  final_score  \
0        1       101  0.994850    5.99           1     1.094850   
3        1       104  0.994850    4.99           1     1.094850   
5        1       106  0.744845    1.99           1     0.744845   
1        1       102  0.082761    3.99           1    -0.417239   
2        1       103  0.082761    2.99           1    -0.417239   

   real_interest  valor_esperado  
0              1        6.558150  
3              1        5.463300  
5              0        1.482242  
1              0       -1.664785  
2              0       -1.247546  

--- MÉTRICAS DE CALIDAD ---
Precision@5: 0.40
Recall@5:    0.67
nDCG@5:      1.00

--- IMPACTO DE NEGOCIO ---
Ingreso Estimado Total (Simulado): $54.85
